# GFP tuned — Fixed Architecture, Data-Tuned GFP Runs (HI-Small)

- **Question:** does the data-tuned GFP sheet `tuned` (64 columns: 48 h windows, simple cycles ≤ 12, bins [2, 4, 6, 8]; `GFP_experiments.ipynb`) beat the paper's V0 sheet (61 columns) when the GNN architecture is held *exactly* fixed?
- **Knobs per run:** `operator` (gin / gat / pna / transformer), `mp` / `readout` edge features as in the operator notebooks, `gfp_variant = 'tuned'`. The swap happens at load time (`gnn_core.load_graphs`): the graph files stay as they are, `edge_attr` becomes 20 baseline + 64 tuned = 84 columns; only the edge projections and the readout's first Linear widen (337 → 340 inputs), nothing else changes. `NODE_ENC` stays `none` — one factor at a time.
- **Everything shared** (model template, training loop, threshold selection, metrics, plots, saving) lives in `gnn_core.py`; this notebook only sets the run list and paths. All eight runs in one session, so the executed notebook holds every training log; each run is compared with the same configuration on V0 in that operator's notebook, on validation F1 (gap < 0.02 = tie).
- **Protocol:** threshold swept on validation each epoch, best-val-F1 checkpoint, test scored once; every metric on train / val / test. `invariant_params` must stay at the family value (GIN 67,587 · GATv2 67,585 · PNA 427,265 · Transformer 133,121).

## 0. Environment (Kaggle)

- Pinned PyTorch / PyG stack known to work on Kaggle GPU — do not change versions or order.
- Then clone (or pull) the public repo so the committed `gnn_core.py` is the one imported.

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric

In [ ]:
import os
import sys

REPO_URL = 'https://github.com/sandrokhizanishvili/gnn_for_fraudulent_patterns.git'
REPO_DIR = '/kaggle/working/gnn_for_fraudulent_patterns'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline
sys.path.append(REPO_DIR)

## 1. Configuration

- The only cell that differs from the operator notebooks: `CONFIGS` lists `(operator, mp, readout, gfp_variant)` per run; all eight runs are active (configs 5 and 4 of every operator).
- `GFP_VARIANT` is fixed for the whole batch (the sheet is swapped in once, at load time); `GFP_DIR`: the Kaggle dataset `hi-small-gfp-tuned` (`Data/gfp_variants/tuned.npy` + `tuned_cols.json`), added as a second input next to `hi-small-gnn`.
- Runs land in `OUT_BASE/<OPERATOR>/<run>/`, mirroring `Outputs/GFP_TUNED/<FAMILY>/` in the repo (separate from the operator batches in `Outputs/<FAMILY>/`); the batch summary is `batch_summary_gfp_tuned.csv` per family.

In [ ]:
# runs of this batch — all eight in one session, so this notebook holds every training log
CONFIGS = [
    dict(operator='gin', mp='base', readout='full', gfp_variant='tuned'),          # GIN-5 + tuned — compare with GIN-5
    dict(operator='gin', mp='full', readout='full', gfp_variant='tuned'),          # GIN-4 + tuned — compare with GIN-4
    dict(operator='gat', mp='base', readout='full', gfp_variant='tuned'),          # GAT-5 + tuned — compare with GAT-5
    dict(operator='gat', mp='full', readout='full', gfp_variant='tuned'),          # GAT-4 + tuned — compare with GAT-4
    dict(operator='pna', mp='base', readout='full', gfp_variant='tuned'),          # PNA-5 + tuned — compare with PNA-5
    dict(operator='pna', mp='full', readout='full', gfp_variant='tuned'),          # PNA-4 + tuned — compare with PNA-4
    dict(operator='transformer', mp='base', readout='full', gfp_variant='tuned'),  # TR-5 + tuned — compare with TR-5
    dict(operator='transformer', mp='full', readout='full', gfp_variant='tuned'),  # TR-4 + tuned — compare with TR-4
]
GFP_VARIANT       = 'tuned'  # 'v0' | 'tuned' — fixed for the whole batch (the sheet is swapped in once, at load time)
MP_DIRECTION      = 'in'     # 'in' | 'bidirectional' — fixed for the whole batch
TEMPORAL_SAMPLING = False    # True samples only edges earlier than the seed (needs pyg-lib)
assert all(cfg['gfp_variant'] == GFP_VARIANT for cfg in CONFIGS), 'one GFP sheet per batch'

# paths
DATA_DIR    = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn'
GFP_DIR     = '/kaggle/input/datasets/sandrokhizanishvili/hi-small-gfp-tuned'   # Data/gfp_variants/ as a separate Kaggle dataset
OUT_BASE    = '/kaggle/working/Outputs/GFP_TUNED'                               # runs go to OUT_BASE/<OPERATOR>/, mirrors Outputs/GFP_TUNED/<FAMILY>/

import gnn_core as core
for cfg in CONFIGS:
    os.makedirs(f"{OUT_BASE}/{cfg['operator'].upper()}", exist_ok=True)
print(f'torch {core.torch.__version__} | device {core.device} | gnn_core from {core.__file__}')

## 2. Load graphs

- Three cumulative PyG snapshots; context edges carry label −1, only `eval_mask` edges are scored.
- `GFP_VARIANT = 'tuned'`: `tuned.npy` is normalised exactly like V0 (log1p → clip at train p1 / p99 → StandardScaler, fit on the train rows; pattern bins untouched) and replaces the GFP block of every snapshot, so `edge_attr` becomes 84 wide.
- `col_sets` maps each knob value to `edge_attr` columns: base = first 20, gfp = next 64, full = all 84.

In [ ]:
graphs, col_sets, node_dim = core.load_graphs(DATA_DIR, GFP_VARIANT, GFP_DIR)

## 3. Model

- Fixed template (`gnn_core.EdgeClassifier`): `node_proj` → 2 message-passing layers (hidden 128, dropout 0.3, residual) → readout MLP on `[h_src ‖ h_dst ‖ e_seed]`; only the conv differs between operators.
- Against the V0 run of the same configuration only the readout's first Linear widens (337 → 340 inputs) and, with `mp = full`, the conv's edge projection (81 → 84 inputs); +384 parameters per widened tensor.
- `invariant_params` leaves those tensors out and must print the family value; PNA also needs the training-graph in-degree histogram.

In [ ]:
cfg = CONFIGS[0]
deg_hist = core.in_degree_histogram(graphs['train']) if cfg['operator'] == 'pna' else None
model = core.build_model(cfg['operator'], node_dim, len(col_sets[cfg['mp']]), len(col_sets[cfg['readout']]),
                         MP_DIRECTION == 'bidirectional', deg_hist)
print(model)
print(f'readout input {model.classifier[0].in_features} | params {core.count_params(model):,} | '
      f'invariant_params {core.invariant_params(model):,}   (config {cfg})')
del model

## 4. Run the batch

- One call per config: seed → model → loaders → 20 epochs → best checkpoint → all splits scored → `results.json`, `history.csv`, `curves.png`, `best.pt`, `predictions.csv` under `OUT_ROOT/<run>/`.
- `predictions.csv`: every evaluated edge of train / val / test with `edge_id` (= row in `Data/edge_features.csv`), `src`, `dst`, `edge_time`, `y`, `prob` (probability) and `pred` (class at the val-chosen threshold) — for the typology / error analyses.
- Per-epoch line: train / val loss, F1 (train at the val threshold), PR-AUC, seconds, RAM; `* new best` marks a checkpoint.

In [ ]:
rows = []
for cfg in CONFIGS:
    print('=' * 90)
    rows.append(core.run_experiment(cfg['operator'], cfg, graphs, col_sets, node_dim,
                                    f"{OUT_BASE}/{cfg['operator'].upper()}", MP_DIRECTION, TEMPORAL_SAMPLING,
                                    gfp_variant=cfg['gfp_variant']))
print('=' * 90)
print('batch complete')

## 5. Comparison across the batch

- `batch_summary_gfp_tuned.csv` per operator holds every `<split>_<metric>` column plus `threshold`, `best_epoch`, `params`, `invariant_params`, `gfp_variant`.
- `invariant_params` must be identical in every row of an operator (asserted).

In [ ]:
for operator in sorted({cfg['operator'] for cfg in CONFIGS}):
    summary = core.summarize_batch([row for row in rows if row['operator'] == operator],
                                   f'{OUT_BASE}/{operator.upper()}', 'batch_summary_gfp_tuned.csv')
    display(summary[['gfp_variant'] + core.DISPLAY_COLS].round(4))

## 6. Download

- Zip `OUT_BASE`; extract into `Outputs/GFP_TUNED/` in the repo and commit the small result files (`best.pt` and `predictions.csv` stay out of git).

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/Outputs_GFP_TUNED', 'zip', OUT_BASE)